# 7. Model Evaluation & Comparison

## Objective

Evaluate and compare the models developed in the previous notebooks.

## Models / Systems Evaluated

- Rating Regression
- Visit Mode Classification
- Baseline Hybrid Recommendation System
- Advanced Recommendation System

## Key Goals

- Compare model performance using appropriate evaluation metrics
- Evaluate recommendation quality and coverage
- Identify the strengths and limitations of each approach
- Select the most suitable models for the final tourism recommendation pipeline

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Load dataset and saved model artifacts

df = pd.read_csv("tourism_cleaned_engineered.csv")

regression_model = joblib.load("best_regression_model.pkl")
regression_features = joblib.load("regression_feature_columns.pkl")

classification_model = joblib.load("best_classification_model.pkl")
classification_features = joblib.load("classification_feature_columns.pkl")

visitmode_label_map = joblib.load("visitmode_label_map.pkl")

print("Dataset Shape:", df.shape)
print("\nRegression Features:")
print(regression_features)

print("\nClassification Features:")
print(classification_features)

print("\nVisitMode Label Map:")
print(visitmode_label_map)

print("\nModels loaded successfully.")

Dataset Shape: (52930, 24)

Regression Features:
['VisitYear', 'VisitMonth', 'AttractionTypeId', 'user_visit_count', 'attraction_avg_rating', 'Country_Albania', 'Country_Algeria', 'Country_Angola', 'Country_Argentina', 'Country_Armenia', 'Country_Asia', 'Country_Australia', 'Country_Austria', 'Country_Azerbaijan', 'Country_Bahamas', 'Country_Bahrain', 'Country_Bangladesh', 'Country_Barbados', 'Country_Belarus', 'Country_Belgium', 'Country_Bhutan', 'Country_Bolivia', 'Country_Bosnia and Herzegovina', 'Country_Botswana', 'Country_Brazil', 'Country_Brunei Darussalam', 'Country_Bulgaria', 'Country_Burkina Faso', 'Country_Cambodia', 'Country_Cameroon', 'Country_Canada', 'Country_Cape Verde', 'Country_Chad', 'Country_Chile', 'Country_China', 'Country_Colombia', 'Country_Costa Rica', 'Country_Croatia', 'Country_Cuba', 'Country_Curacao', 'Country_Cyprus', 'Country_Czech Republic', 'Country_Denmark', 'Country_Dominican Republic', 'Country_East Timor', 'Country_Ecuador', 'Country_Egypt', 'Countr

In [3]:
# Prepare data for regression evaluation

X_regression = df.reindex(
    columns=regression_features,
    fill_value=0
)

y_regression = df["Rating"]

print("Regression Evaluation Data")
print("--------------------------")
print("Feature Matrix Shape:", X_regression.shape)
print("Target Shape:", y_regression.shape)
print("Target Mean:", round(y_regression.mean(), 4))
print("Missing Values:", X_regression.isna().sum().sum())

Regression Evaluation Data
--------------------------
Feature Matrix Shape: (52930, 159)
Target Shape: (52930,)
Target Mean: 4.1577
Missing Values: 0


In [4]:
# Generate regression predictions

y_pred_regression = regression_model.predict(X_regression)

print("Regression Predictions Generated")
print("-------------------------------")
print("Prediction Shape:", y_pred_regression.shape)
print("Prediction Mean:", round(y_pred_regression.mean(), 4))
print("Prediction Min:", round(y_pred_regression.min(), 4))
print("Prediction Max:", round(y_pred_regression.max(), 4))

Regression Predictions Generated
-------------------------------
Prediction Shape: (52930,)
Prediction Mean: 4.1585
Prediction Min: 1.5585
Prediction Max: 5.1129


In [6]:
# Generate regression predictions

y_pred_regression = regression_model.predict(X_regression)

print("Regression Predictions Generated")
print("-------------------------------")
print("Prediction Shape:", y_pred_regression.shape)
print("Prediction Mean:", round(y_pred_regression.mean(), 4))
print("Prediction Min:", round(y_pred_regression.min(), 4))
print("Prediction Max:", round(y_pred_regression.max(), 4))

Regression Predictions Generated
-------------------------------
Prediction Shape: (52930,)
Prediction Mean: 4.1585
Prediction Min: 1.5585
Prediction Max: 5.1129


In [7]:
# Calculate regression evaluation metrics

mse = mean_squared_error(y_regression, y_pred_regression)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_regression, y_pred_regression)
r2 = r2_score(y_regression, y_pred_regression)

print("Regression Evaluation Metrics")
print("----------------------------")
print("MSE :", round(mse, 4))
print("RMSE:", round(rmse, 4))
print("MAE :", round(mae, 4))
print("R²  :", round(r2, 4))

Regression Evaluation Metrics
----------------------------
MSE : 0.8263
RMSE: 0.909
MAE : 0.7138
R²  : 0.1227


In [9]:
from sklearn.model_selection import train_test_split

# Recreate the exact raw feature setup from Notebook 2

feature_cols = [
    'VisitYear',
    'VisitMonth',
    'AttractionTypeId',
    'Continent',
    'Region',
    'Country',
    'VisitMode',
    'user_visit_count'
]

X = df[feature_cols + ['UserId', 'AttractionId']].copy()
y = df['Rating']

# Exact same 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Exact Regression Split")
print("----------------------")
print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)
print("Train Target:", y_train.shape)
print("Test Target :", y_test.shape)

Exact Regression Split
----------------------
Train Shape: (42344, 10)
Test Shape : (10586, 10)
Train Target: (42344,)
Test Target : (10586,)


In [10]:
# Recreate training-only averages exactly as in Notebook 2

user_avg_rating_map = y_train.groupby(X_train["UserId"]).mean()
attraction_avg_rating_map = y_train.groupby(X_train["AttractionId"]).mean()
global_avg_rating = y_train.mean()

# Apply training-derived mappings to both train and test
X_train["user_avg_rating"] = (
    X_train["UserId"].map(user_avg_rating_map).fillna(global_avg_rating)
)

X_train["attraction_avg_rating"] = (
    X_train["AttractionId"].map(attraction_avg_rating_map).fillna(global_avg_rating)
)

X_test["user_avg_rating"] = (
    X_test["UserId"].map(user_avg_rating_map).fillna(global_avg_rating)
)

X_test["attraction_avg_rating"] = (
    X_test["AttractionId"].map(attraction_avg_rating_map).fillna(global_avg_rating)
)

print("Training-only feature engineering completed.")
print("Global Average Rating:", round(global_avg_rating, 4))
print("Train Missing Values:", X_train.isna().sum().sum())
print("Test Missing Values :", X_test.isna().sum().sum())

Training-only feature engineering completed.
Global Average Rating: 4.1574
Train Missing Values: 0
Test Missing Values : 0


In [11]:
# Recreate the exact encoding used in Notebook 2

X_train = X_train.drop(columns=["Continent", "Region"])
X_test = X_test.drop(columns=["Continent", "Region"])

cat_cols = ["Country", "VisitMode"]

X_train_enc = pd.get_dummies(
    X_train,
    columns=cat_cols,
    drop_first=True
)

X_test_enc = pd.get_dummies(
    X_test,
    columns=cat_cols,
    drop_first=True
)

# Align test columns exactly with training columns
X_train_enc, X_test_enc = X_train_enc.align(
    X_test_enc,
    join="left",
    axis=1,
    fill_value=0
)

print("Encoding Completed")
print("------------------")
print("Train Encoded Shape:", X_train_enc.shape)
print("Test Encoded Shape :", X_test_enc.shape)
print("Missing Values - Train:", X_train_enc.isna().sum().sum())
print("Missing Values - Test :", X_test_enc.isna().sum().sum())

Encoding Completed
------------------
Train Encoded Shape: (42344, 162)
Test Encoded Shape : (10586, 162)
Missing Values - Train: 0
Missing Values - Test : 0


In [13]:
# Freshly recreate the exact regression preprocessing

feature_cols = [
    "VisitYear",
    "VisitMonth",
    "AttractionTypeId",
    "Continent",
    "Region",
    "Country",
    "VisitMode",
    "user_visit_count"
]

# Fresh X and y
X = df[feature_cols + ["UserId", "AttractionId"]].copy()
y = df["Rating"]

# Exact same split as Notebook 2
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Training-only averages
user_avg_rating_map = y_train.groupby(X_train["UserId"]).mean()
attraction_avg_rating_map = y_train.groupby(X_train["AttractionId"]).mean()
global_avg_rating = y_train.mean()

X_train["user_avg_rating"] = X_train["UserId"].map(
    user_avg_rating_map
).fillna(global_avg_rating)

X_train["attraction_avg_rating"] = X_train["AttractionId"].map(
    attraction_avg_rating_map
).fillna(global_avg_rating)

X_test["user_avg_rating"] = X_test["UserId"].map(
    user_avg_rating_map
).fillna(global_avg_rating)

X_test["attraction_avg_rating"] = X_test["AttractionId"].map(
    attraction_avg_rating_map
).fillna(global_avg_rating)

# Remove IDs
X_train = X_train.drop(columns=["UserId", "AttractionId"])
X_test = X_test.drop(columns=["UserId", "AttractionId"])

# Remove Continent and Region
X_train = X_train.drop(columns=["Continent", "Region"])
X_test = X_test.drop(columns=["Continent", "Region"])

# One-hot encoding
cat_cols = ["Country", "VisitMode"]

X_train_enc = pd.get_dummies(
    X_train,
    columns=cat_cols,
    drop_first=True
)

X_test_enc = pd.get_dummies(
    X_test,
    columns=cat_cols,
    drop_first=True
)

# Align exactly like Notebook 2
X_train_enc, X_test_enc = X_train_enc.align(
    X_test_enc,
    join="left",
    axis=1,
    fill_value=0
)

print("Regression preprocessing recreated successfully")
print("------------------------------------------------")
print("Train Shape:", X_train_enc.shape)
print("Test Shape :", X_test_enc.shape)
print("Train Missing Values:", X_train_enc.isna().sum().sum())
print("Test Missing Values :", X_test_enc.isna().sum().sum())

Regression preprocessing recreated successfully
------------------------------------------------
Train Shape: (42344, 160)
Test Shape : (10586, 160)
Train Missing Values: 0
Test Missing Values : 0


In [14]:
# Prepare the exact final feature set used by the saved regression model

X_test_final = X_test_enc.drop(
    columns=["user_avg_rating"]
)

# Ensure exact feature order expected by the saved model
X_test_final = X_test_final.reindex(
    columns=regression_features,
    fill_value=0
)

# Generate predictions on the held-out test set
y_pred_test = regression_model.predict(X_test_final)

print("Final Test Set Ready")
print("--------------------")
print("Test Feature Shape:", X_test_final.shape)
print("Actual Target Shape:", y_test.shape)
print("Prediction Shape:", y_pred_test.shape)
print("Prediction Mean:", round(y_pred_test.mean(), 4))
print("Prediction Min :", round(y_pred_test.min(), 4))
print("Prediction Max :", round(y_pred_test.max(), 4))

Final Test Set Ready
--------------------
Test Feature Shape: (10586, 159)
Actual Target Shape: (10586,)
Prediction Shape: (10586,)
Prediction Mean: 4.1607
Prediction Min : 1.0483
Prediction Max : 5.1375


In [15]:
# Evaluate regression model on the exact held-out test set

mse_test = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print("Final Regression Test Metrics")
print("-----------------------------")
print("MSE :", round(mse_test, 4))
print("RMSE:", round(rmse_test, 4))
print("MAE :", round(mae_test, 4))
print("R²  :", round(r2_test, 4))

Final Regression Test Metrics
-----------------------------
MSE : 0.8239
RMSE: 0.9077
MAE : 0.7105
R²  : 0.1253


In [16]:
# Store final regression results for model comparison

regression_results = {
    "Model": "XGBoost Regression",
    "MSE": mse_test,
    "RMSE": rmse_test,
    "MAE": mae_test,
    "R2": r2_test
}

print("Regression results stored successfully.")
print(regression_results)

Regression results stored successfully.
{'Model': 'XGBoost Regression', 'MSE': 0.8238537311553955, 'RMSE': np.float64(0.9076638866647695), 'MAE': 0.7105441093444824, 'R2': 0.12525004148483276}


In [18]:
# Recreate the exact classification setup from Notebook 3

classification_feature_cols = [
    "VisitYear",
    "VisitMonth",
    "AttractionTypeId",
    "Continent",
    "Region",
    "Country",
    "Rating",
    "user_visit_count"
]

X_cls = df[
    classification_feature_cols + ["UserId", "AttractionId"]
].copy()

y_cls = df["VisitMode"]

# Exact split used in Notebook 3
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

print("Exact Classification Split")
print("--------------------------")
print("Train Shape:", X_train_cls.shape)
print("Test Shape :", X_test_cls.shape)
print("Train Target:", y_train_cls.shape)
print("Test Target :", y_test_cls.shape)

Exact Classification Split
--------------------------
Train Shape: (42344, 10)
Test Shape : (10586, 10)
Train Target: (42344,)
Test Target : (10586,)


In [19]:
# Recreate the exact preprocessing used in Notebook 3

# Training-only attraction average
attraction_avg_rating_map_clf = (
    X_train_cls.groupby("AttractionId")["Rating"].mean()
)

global_avg_rating_clf = X_train_cls["Rating"].mean()

X_train_cls["attraction_avg_rating"] = (
    X_train_cls["AttractionId"]
    .map(attraction_avg_rating_map_clf)
    .fillna(global_avg_rating_clf)
)

X_test_cls["attraction_avg_rating"] = (
    X_test_cls["AttractionId"]
    .map(attraction_avg_rating_map_clf)
    .fillna(global_avg_rating_clf)
)

# Remove IDs
X_train_cls = X_train_cls.drop(columns=["UserId", "AttractionId"])
X_test_cls = X_test_cls.drop(columns=["UserId", "AttractionId"])

# Drop Continent and Region
X_train_cls = X_train_cls.drop(columns=["Continent", "Region"])
X_test_cls = X_test_cls.drop(columns=["Continent", "Region"])

# Encode Country only
X_train_cls_enc = pd.get_dummies(
    X_train_cls,
    columns=["Country"],
    drop_first=True
)

X_test_cls_enc = pd.get_dummies(
    X_test_cls,
    columns=["Country"],
    drop_first=True
)

# Align test features with training features
X_train_cls_enc, X_test_cls_enc = X_train_cls_enc.align(
    X_test_cls_enc,
    join="left",
    axis=1,
    fill_value=0
)

print("Classification preprocessing completed")
print("---------------------------------------")
print("Train Encoded Shape:", X_train_cls_enc.shape)
print("Test Encoded Shape :", X_test_cls_enc.shape)
print("Train Missing Values:", X_train_cls_enc.isna().sum().sum())
print("Test Missing Values :", X_test_cls_enc.isna().sum().sum())

Classification preprocessing completed
---------------------------------------
Train Encoded Shape: (42344, 157)
Test Encoded Shape : (10586, 157)
Train Missing Values: 0
Test Missing Values : 0


In [20]:
# Prepare exact feature set expected by the saved classification model

X_test_cls_final = X_test_cls_enc.reindex(
    columns=classification_features,
    fill_value=0
)

# Encode target labels using the saved label mapping
y_test_cls_encoded = y_test_cls.map(visitmode_label_map)

# Generate predictions
y_pred_cls = classification_model.predict(X_test_cls_final)

print("Classification Test Predictions Generated")
print("-----------------------------------------")
print("Test Feature Shape:", X_test_cls_final.shape)
print("Actual Target Shape:", y_test_cls_encoded.shape)
print("Prediction Shape:", y_pred_cls.shape)
print("Unique Predicted Classes:", np.unique(y_pred_cls))

Classification Test Predictions Generated
-----------------------------------------
Test Feature Shape: (10586, 157)
Actual Target Shape: (10586,)
Prediction Shape: (10586,)
Unique Predicted Classes: [0 1 2 3 4]


In [21]:
# Evaluate classification model on the exact held-out test set

accuracy_test = accuracy_score(
    y_test_cls_encoded,
    y_pred_cls
)

precision_test = precision_score(
    y_test_cls_encoded,
    y_pred_cls,
    average="macro",
    zero_division=0
)

recall_test = recall_score(
    y_test_cls_encoded,
    y_pred_cls,
    average="macro",
    zero_division=0
)

f1_test = f1_score(
    y_test_cls_encoded,
    y_pred_cls,
    average="macro",
    zero_division=0
)

print("Final Classification Test Metrics")
print("---------------------------------")
print("Accuracy :", round(accuracy_test, 4))
print("Precision:", round(precision_test, 4))
print("Recall   :", round(recall_test, 4))
print("Macro F1 :", round(f1_test, 4))

Final Classification Test Metrics
---------------------------------
Accuracy : 0.4013
Precision: 0.3278
Recall   : 0.3961
Macro F1 : 0.3187


In [22]:
# Store final classification results for model comparison

classification_results = {
    "Model": "XGBoost Classification",
    "Accuracy": accuracy_test,
    "Precision": precision_test,
    "Recall": recall_test,
    "Macro F1": f1_test
}

print("Classification results stored successfully.")
print(classification_results)

Classification results stored successfully.
{'Model': 'XGBoost Classification', 'Accuracy': 0.40128471566219537, 'Precision': 0.3277822920507899, 'Recall': 0.3960831393506691, 'Macro F1': 0.3187083864769465}


In [23]:
# Check whether recommendation functions are available in the current notebook

print("Recommendation Functions Check")
print("-------------------------------")

print(
    "Baseline hybrid_recommendation:",
    "Available" if "hybrid_recommendation" in globals() else "Not Available"
)

print(
    "Advanced advanced_recommendation:",
    "Available" if "advanced_recommendation" in globals() else "Not Available"
)

print(
    "Content similarity:",
    "Available" if "content_similarity_df" in globals() else "Not Available"
)

print(
    "Item similarity:",
    "Available" if "item_similarity_df" in globals() else "Not Available"
)

Recommendation Functions Check
-------------------------------
Baseline hybrid_recommendation: Not Available
Advanced advanced_recommendation: Not Available
Content similarity: Not Available
Item similarity: Not Available


In [24]:
# Load recommendation-system data and recreate Notebook 6 setup

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

# Load full attraction catalog
item = pd.read_excel(
    "Tourism_Analytics/data/Additional_Data_for_Attraction_Sites/Updated_Item.xlsx"
)

type_df = pd.read_excel(
    "Tourism_Analytics/data/Type.xlsx"
)

item_full = item.merge(
    type_df,
    on="AttractionTypeId",
    how="left"
)

# User-item matrix for collaborative filtering
user_item_matrix = df.pivot_table(
    index="UserId",
    columns="AttractionId",
    values="Rating",
    aggfunc="mean",
    fill_value=0
)

item_user_matrix = user_item_matrix.T

collaborative_similarity = cosine_similarity(
    item_user_matrix
)

collaborative_similarity_df = pd.DataFrame(
    collaborative_similarity,
    index=item_user_matrix.index,
    columns=item_user_matrix.index
)

print("Recommendation setup completed.")
print("-------------------------------")
print("Full attraction catalog:", item_full.shape)
print("User-Item Matrix:", user_item_matrix.shape)
print("Collaborative Similarity:", collaborative_similarity_df.shape)

Recommendation setup completed.
-------------------------------
Full attraction catalog: (1698, 6)
User-Item Matrix: (33530, 30)
Collaborative Similarity: (30, 30)


In [25]:
# Recreate Notebook 6 content-based similarity

# Normalize attraction type names
type_lookup_map = (
    type_df.set_index("AttractionTypeId")["AttractionType"]
    .to_dict()
)

def normalize_type(val):
    val_str = str(val).strip()

    if val_str.isdigit():
        return type_lookup_map.get(int(val_str), "Other")

    return val_str


item_full["AttractionType_clean"] = (
    item_full["AttractionTypeId"].apply(normalize_type)
)

# Extract city from attraction name
item_full["city_from_name"] = item_full["Attraction"].apply(
    lambda x: x.split(" - ")[-1].strip()
    if isinstance(x, str) and " - " in x
    else "Unknown"
)

# Treat unknown / placeholder city values as missing
item_full["city_from_name"] = item_full["city_from_name"].replace(
    ["Unknown", "-", ""],
    np.nan
)

# One-hot encode type and city
type_dummies = pd.get_dummies(
    item_full["AttractionType_clean"].fillna("Other").astype(str)
)

city_dummies = pd.get_dummies(
    item_full["city_from_name"].fillna("Unknown").astype(str)
)

# Similarity matrices
type_sim = cosine_similarity(type_dummies)
city_sim = cosine_similarity(city_dummies)

# Notebook 6 weighting
combined_sim = (
    0.3 * type_sim +
    0.7 * city_sim
)

content_similarity_df = pd.DataFrame(
    combined_sim,
    index=item_full["AttractionId"],
    columns=item_full["AttractionId"]
)

print("Content-Based Similarity Created")
print("--------------------------------")
print("Type matrix:", type_dummies.shape)
print("City matrix:", city_dummies.shape)
print("Combined similarity:", content_similarity_df.shape)

Content-Based Similarity Created
--------------------------------
Type matrix: (1698, 22)
City matrix: (1698, 417)
Combined similarity: (1698, 1698)


In [26]:
# Recreate Notebook 6 attraction popularity scores

attraction_popularity = (
    df.groupby("AttractionId")
    .agg(
        visit_count=("TransactionId", "count"),
        unique_users=("UserId", "nunique"),
        avg_rating=("Rating", "mean")
    )
    .reset_index()
)

# Scale popularity features
scaler_popularity = StandardScaler()

attraction_popularity[
    ["visit_count_scaled", "unique_users_scaled", "avg_rating_scaled"]
] = scaler_popularity.fit_transform(
    attraction_popularity[
        ["visit_count", "unique_users", "avg_rating"]
    ]
)

# Combined popularity score
attraction_popularity["popularity_score"] = (
    0.5 * attraction_popularity["visit_count_scaled"] +
    0.3 * attraction_popularity["unique_users_scaled"] +
    0.2 * attraction_popularity["avg_rating_scaled"]
)

# Normalize popularity to 0-1
min_pop = attraction_popularity["popularity_score"].min()
max_pop = attraction_popularity["popularity_score"].max()

attraction_popularity["popularity_normalized"] = (
    (attraction_popularity["popularity_score"] - min_pop)
    / (max_pop - min_pop)
)

print("Attraction Popularity Created")
print("-----------------------------")
print("Attractions with popularity scores:",
      len(attraction_popularity))

print("\nTop 5 Popular Attractions:")
print(
    attraction_popularity
    .sort_values("popularity_score", ascending=False)
    [["AttractionId", "visit_count",
      "unique_users", "avg_rating",
      "popularity_score", "popularity_normalized"]]
    .head(5)
)

Attraction Popularity Created
-----------------------------
Attractions with popularity scores: 30

Top 5 Popular Attractions:
   AttractionId  visit_count  unique_users  avg_rating  popularity_score  \
2           640        13198         11487    4.267086          3.438255   
9           841         6429          5605    4.646601          1.710347   
6           748         5815          5415    4.157524          1.261456   
8           824         3359          3183    4.219411          0.573977   
5           737         3352          3156    4.194809          0.553632   

   popularity_normalized  
2               1.000000  
9               0.594857  
6               0.489605  
8               0.328412  
5               0.323641  


In [27]:
# Recreate Notebook 6 user-specific recommendation weights

def get_recommendation_weights(interactions):
    if interactions <= 2:
        return {
            "collaborative": 0.20,
            "content": 0.30,
            "popularity": 0.50
        }
    elif interactions <= 5:
        return {
            "collaborative": 0.35,
            "content": 0.40,
            "popularity": 0.25
        }
    else:
        return {
            "collaborative": 0.45,
            "content": 0.40,
            "popularity": 0.15
        }


user_interaction_counts = (
    df.groupby("UserId")["AttractionId"]
    .count()
)

user_weight_map = {
    user_id: get_recommendation_weights(count)
    for user_id, count in user_interaction_counts.items()
}

print("User-Specific Weights Created")
print("------------------------------")

print("Total users:", len(user_weight_map))

print("\nExample weights:")

for user_id in [20, 14, 16, 60799]:
    if user_id in user_weight_map:
        print(
            user_id,
            "->",
            user_weight_map[user_id]
        )

User-Specific Weights Created
------------------------------
Total users: 33530

Example weights:
20 -> {'collaborative': 0.2, 'content': 0.3, 'popularity': 0.5}
14 -> {'collaborative': 0.35, 'content': 0.4, 'popularity': 0.25}
16 -> {'collaborative': 0.45, 'content': 0.4, 'popularity': 0.15}
60799 -> {'collaborative': 0.45, 'content': 0.4, 'popularity': 0.15}


In [28]:
# Recreate Notebook 6 advanced recommendation function

def advanced_recommendation(user_id, top_n=5):

    if user_id not in user_weight_map:
        return []

    weights = user_weight_map[user_id]

    user_history_df = df[
        df["UserId"] == user_id
    ][["AttractionId", "Rating"]].copy()

    visited_attractions = set(
        user_history_df["AttractionId"]
    )

    content_scores = {}
    collaborative_scores = {}
    popularity_scores_dict = {}

    # Generate scores from user's history
    for _, row in user_history_df.iterrows():

        attraction_id = row["AttractionId"]
        rating = row["Rating"]

        # Content-based scores
        if attraction_id in content_similarity_df.index:

            similarities = content_similarity_df.loc[
                attraction_id
            ]

            for candidate_id, similarity in similarities.items():

                if candidate_id not in visited_attractions:

                    content_scores[candidate_id] = (
                        content_scores.get(candidate_id, 0)
                        + rating * similarity
                    )

        # Collaborative filtering scores
        if attraction_id in collaborative_similarity_df.index:

            similarities = collaborative_similarity_df.loc[
                attraction_id
            ]

            for candidate_id, similarity in similarities.items():

                if candidate_id not in visited_attractions:

                    collaborative_scores[candidate_id] = (
                        collaborative_scores.get(candidate_id, 0)
                        + rating * similarity
                    )

    # Popularity scores
    for attraction_id, score in popularity_map.items():

        if attraction_id not in visited_attractions:
            popularity_scores_dict[attraction_id] = score

    # Normalize score dictionaries
    def normalize_scores(scores):

        if not scores:
            return {}

        max_score = max(scores.values())

        if max_score == 0:
            return {
                k: 0 for k in scores
            }

        return {
            k: v / max_score
            for k, v in scores.items()
        }

    content_scores = normalize_scores(content_scores)
    collaborative_scores = normalize_scores(
        collaborative_scores
    )
    popularity_scores_dict = normalize_scores(
        popularity_scores_dict
    )

    # Combine scores
    all_candidates = (
        set(content_scores)
        | set(collaborative_scores)
        | set(popularity_scores_dict)
    )

    final_scores = {}

    for attraction_id in all_candidates:

        final_scores[attraction_id] = (
            weights["collaborative"]
            * collaborative_scores.get(attraction_id, 0)
            +
            weights["content"]
            * content_scores.get(attraction_id, 0)
            +
            weights["popularity"]
            * popularity_scores_dict.get(attraction_id, 0)
        )

    # Sort recommendations
    top_recommendations = sorted(
        final_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

    # Convert IDs to attraction names
    recommendation_df = pd.DataFrame(
        top_recommendations,
        columns=["AttractionId", "Score"]
    )

    recommendation_df = recommendation_df.merge(
        item_full[
            ["AttractionId", "Attraction"]
        ],
        on="AttractionId",
        how="left"
    )

    return recommendation_df[
        ["AttractionId", "Attraction", "Score"]
    ]

In [29]:
# Compare Baseline Hybrid vs Advanced Recommendation
# User 60799

test_user = 60799

print("User:", test_user)
print("=" * 70)

print("\nAdvanced Recommendation:")
advanced_recs = advanced_recommendation(test_user, top_n=5)
print(advanced_recs)

print("\nAdvanced recommendations generated:",
      len(advanced_recs))

User: 60799

Advanced Recommendation:


NameError: name 'popularity_map' is not defined

In [30]:
# Create normalized popularity map for the recommender

popularity_map = (
    attraction_popularity
    .set_index("AttractionId")["popularity_normalized"]
    .to_dict()
)

print("Popularity map created.")
print("Attractions with popularity scores:", len(popularity_map))

Popularity map created.
Attractions with popularity scores: 30


In [31]:
# Test Advanced Recommendation

test_user = 60799

print("User:", test_user)
print("=" * 70)

print("\nAdvanced Recommendation:")
advanced_recs = advanced_recommendation(test_user, top_n=5)

print(advanced_recs)

print("\nAdvanced recommendations generated:",
      len(advanced_recs))

User: 60799

Advanced Recommendation:
   AttractionId                          Attraction     Score
0           824                      Uluwatu Temple  0.827781
1           737                    Tanah Lot Temple  0.820490
2           749                Tegenungan Waterfall  0.727623
3           841                       Waterbom Bali  0.711693
4           888  Bromo Tengger Semeru National Park  0.651773

Advanced recommendations generated: 5


In [32]:
# Restore the exact Notebook 6 user weights and advanced recommender

# User history
user_history = df.groupby("UserId").agg(
    interaction_count=("TransactionId", "count"),
    unique_attractions=("AttractionId", "nunique")
).reset_index()

user_history["history_strength"] = (
    user_history["interaction_count"]
    + user_history["unique_attractions"]
)

# Exact recommendation weights
def get_recommendation_weights(history_count):
    if history_count <= 2:
        return 0.20, 0.30, 0.50
    elif history_count <= 5:
        return 0.35, 0.40, 0.25
    else:
        return 0.45, 0.40, 0.15

user_history[
    ["cf_weight", "content_weight", "popularity_weight"]
] = user_history["interaction_count"].apply(
    lambda x: pd.Series(get_recommendation_weights(x))
)

# Exact lookup map
user_weight_map = user_history.set_index("UserId")[
    ["cf_weight", "content_weight", "popularity_weight"]
].to_dict("index")


# Exact Advanced Recommendation function
def advanced_recommendation(user_id, top_n=5):

    if user_id not in user_weight_map:
        return []

    weights = user_weight_map[user_id]

    cf_weight = weights["cf_weight"]
    content_weight = weights["content_weight"]
    popularity_weight = weights["popularity_weight"]

    user_history_df = df[df["UserId"] == user_id]
    visited_attractions = set(user_history_df["AttractionId"])

    content_scores = {}
    collaborative_scores = {}
    popularity_scores_dict = {}

    for _, row in user_history_df.iterrows():

        attraction_id = row["AttractionId"]
        rating = row["Rating"]

        if attraction_id in content_similarity_df.index:

            similarities = content_similarity_df.loc[attraction_id]

            for candidate_id, similarity in similarities.items():

                if candidate_id not in visited_attractions:
                    content_scores[candidate_id] = (
                        content_scores.get(candidate_id, 0)
                        + rating * similarity
                    )

        if attraction_id in collaborative_similarity_df.index:

            similarities = collaborative_similarity_df.loc[attraction_id]

            for candidate_id, similarity in similarities.items():

                if candidate_id not in visited_attractions:
                    collaborative_scores[candidate_id] = (
                        collaborative_scores.get(candidate_id, 0)
                        + rating * similarity
                    )

    for candidate_id, score in popularity_map.items():

        if candidate_id not in visited_attractions:
            popularity_scores_dict[candidate_id] = score

    def normalize_scores(score_dict):

        if not score_dict:
            return {}

        max_score = max(score_dict.values())

        if max_score == 0:
            return {k: 0 for k in score_dict}

        return {
            k: v / max_score
            for k, v in score_dict.items()
        }

    content_scores = normalize_scores(content_scores)
    collaborative_scores = normalize_scores(collaborative_scores)
    popularity_scores_dict = normalize_scores(popularity_scores_dict)

    all_candidates = (
        set(content_scores)
        | set(collaborative_scores)
        | set(popularity_scores_dict)
    )

    final_scores = {}

    for candidate_id in all_candidates:

        final_scores[candidate_id] = (
            cf_weight * collaborative_scores.get(candidate_id, 0)
            + content_weight * content_scores.get(candidate_id, 0)
            + popularity_weight * popularity_scores_dict.get(candidate_id, 0)
        )

    recommendations = sorted(
        final_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

    attraction_name_map = item_full.set_index(
        "AttractionId"
    )["Attraction"].to_dict()

    return [
        {
            "AttractionId": attraction_id,
            "Attraction": attraction_name_map.get(
                attraction_id, "Unknown"
            ),
            "Score": round(score, 4)
        }
        for attraction_id, score in recommendations
    ]


print("Exact Notebook 6 Advanced Recommender restored.")
print("Users covered:", len(user_weight_map))

Exact Notebook 6 Advanced Recommender restored.
Users covered: 33530


In [33]:
# Final Advanced Recommendation Test

test_user = 60799

print("Test User:", test_user)
print("Interactions:", df[df["UserId"] == test_user].shape[0])
print("\nAdvanced Recommendations")
print("-" * 60)

advanced_recs = advanced_recommendation(
    user_id=test_user,
    top_n=5
)

for i, rec in enumerate(advanced_recs, 1):
    print(
        f"{i}. {rec['Attraction']} "
        f"(ID: {rec['AttractionId']}, Score: {rec['Score']})"
    )

Test User: 60799
Interactions: 59

Advanced Recommendations
------------------------------------------------------------
1. Uluwatu Temple (ID: 824, Score: 0.8278)
2. Tanah Lot Temple (ID: 737, Score: 0.8205)
3. Tegenungan Waterfall (ID: 749, Score: 0.7276)
4. Waterbom Bali (ID: 841, Score: 0.7117)
5. Bromo Tengger Semeru National Park (ID: 888, Score: 0.6518)


In [34]:
# Recreate Notebook 6 final content-based similarity exactly

item_full["city_from_name"] = item_full["Attraction"].apply(
    lambda x: x.split(" - ")[-1].strip()
    if isinstance(x, str) and " - " in x
    else "Unknown"
)

item_full["city_from_name"] = (
    item_full["city_from_name"]
    .replace({"Unknown": None, "-": None})
    .str.strip()
    .replace("", None)
)

type_dummies = pd.get_dummies(
    item_full["AttractionType_clean"]
)

city_dummies = pd.get_dummies(
    item_full["city_from_name"]
)

type_sim = cosine_similarity(type_dummies)
city_sim = cosine_similarity(city_dummies)

combined_sim = (
    0.3 * type_sim
    + 0.7 * city_sim
)

content_similarity_df = pd.DataFrame(
    combined_sim,
    index=item_full["AttractionId"],
    columns=item_full["AttractionId"]
)

print("Exact Notebook 6 Content Similarity Restored")
print("Type matrix:", type_dummies.shape)
print("City matrix:", city_dummies.shape)
print("Combined similarity:", content_similarity_df.shape)

Exact Notebook 6 Content Similarity Restored
Type matrix: (1698, 22)
City matrix: (1698, 416)
Combined similarity: (1698, 1698)


In [35]:
# Re-test Advanced Recommendation after exact similarity restoration

test_user = 60799

print("Test User:", test_user)
print("Interactions:", df[df["UserId"] == test_user].shape[0])
print("\nAdvanced Recommendations")
print("-" * 60)

advanced_recs = advanced_recommendation(
    user_id=test_user,
    top_n=5
)

for i, rec in enumerate(advanced_recs, 1):
    print(
        f"{i}. {rec['Attraction']} "
        f"(ID: {rec['AttractionId']}, Score: {rec['Score']})"
    )

Test User: 60799
Interactions: 59

Advanced Recommendations
------------------------------------------------------------
1. Mount Semeru Volcano (ID: 947, Score: 0.5619)
2. Uluwatu Temple (ID: 824, Score: 0.5328)
3. Tanah Lot Temple (ID: 737, Score: 0.5255)
4. Tegenungan Waterfall (ID: 749, Score: 0.4327)
5. Waterbom Bali (ID: 841, Score: 0.4167)


In [36]:
# Recreate Notebook 4 baseline hybrid recommender

def baseline_hybrid_recommendation(user_id, top_n=5, alpha=0.5):

    user_history_df = df[df["UserId"] == user_id]
    visited_attractions = set(user_history_df["AttractionId"])

    content_scores = {}
    collaborative_scores = {}

    for _, row in user_history_df.iterrows():

        attraction_id = row["AttractionId"]
        rating = row["Rating"]

        # Content-based signal
        if attraction_id in content_similarity_df.index:

            similarities = content_similarity_df.loc[attraction_id]

            for candidate_id, similarity in similarities.items():

                if candidate_id not in visited_attractions:
                    content_scores[candidate_id] = (
                        content_scores.get(candidate_id, 0)
                        + rating * similarity
                    )

        # Collaborative filtering signal
        if attraction_id in collaborative_similarity_df.index:

            similarities = collaborative_similarity_df.loc[attraction_id]

            for candidate_id, similarity in similarities.items():

                if candidate_id not in visited_attractions:
                    collaborative_scores[candidate_id] = (
                        collaborative_scores.get(candidate_id, 0)
                        + rating * similarity
                    )

    # Normalize each signal
    def normalize_scores(score_dict):

        if not score_dict:
            return {}

        max_score = max(score_dict.values())

        if max_score == 0:
            return {k: 0 for k in score_dict}

        return {
            k: v / max_score
            for k, v in score_dict.items()
        }

    content_scores = normalize_scores(content_scores)
    collaborative_scores = normalize_scores(collaborative_scores)

    # Combine CF + Content-Based
    all_candidates = (
        set(content_scores)
        | set(collaborative_scores)
    )

    hybrid_scores = {}

    for candidate_id in all_candidates:

        hybrid_scores[candidate_id] = (
            alpha * collaborative_scores.get(candidate_id, 0)
            + (1 - alpha) * content_scores.get(candidate_id, 0)
        )

    recommendations = sorted(
        hybrid_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

    attraction_name_map = item_full.set_index(
        "AttractionId"
    )["Attraction"].to_dict()

    return [
        {
            "AttractionId": attraction_id,
            "Attraction": attraction_name_map.get(
                attraction_id, "Unknown"
            ),
            "Score": round(score, 4)
        }
        for attraction_id, score in recommendations
    ]


print("Notebook 4 baseline hybrid function recreated successfully.")

Notebook 4 baseline hybrid function recreated successfully.


In [37]:
# Test Baseline Hybrid Recommendation

test_user = 60799

baseline_recs = baseline_hybrid_recommendation(
    user_id=test_user,
    top_n=5,
    alpha=0.5
)

print("Test User:", test_user)
print("Interactions:", len(df[df["UserId"] == test_user]))
print()
print("Notebook 4 - Baseline Hybrid")
print("-" * 44)

for i, rec in enumerate(baseline_recs, 1):
    print(
        f"{i}. {rec['Attraction']} "
        f"(ID: {rec['AttractionId']}, Score: {rec['Score']})"
    )

Test User: 60799
Interactions: 59

Notebook 4 - Baseline Hybrid
--------------------------------------------
1. Mount Semeru Volcano (ID: 947, Score: 0.6324)
2. Uluwatu Temple (ID: 824, Score: 0.5)
3. Tanah Lot Temple (ID: 737, Score: 0.4932)
4. Tegenungan Waterfall (ID: 749, Score: 0.4248)
5. Bromo Tengger Semeru National Park (ID: 888, Score: 0.3467)


In [38]:
# Baseline vs Advanced Recommendation Comparison

advanced_recs = advanced_recommendation(
    user_id=test_user,
    top_n=5
)

baseline_ids = [rec["AttractionId"] for rec in baseline_recs]
advanced_ids = [rec["AttractionId"] for rec in advanced_recs]

common_ids = set(baseline_ids) & set(advanced_ids)
new_in_advanced = set(advanced_ids) - set(baseline_ids)
removed_from_baseline = set(baseline_ids) - set(advanced_ids)

overlap_percentage = (
    len(common_ids) / len(baseline_ids) * 100
    if baseline_ids else 0
)

print("Recommendation Comparison")
print("=" * 50)

print("Baseline recommendations:", baseline_ids)
print("Advanced recommendations:", advanced_ids)
print()

print("Common recommendations:", len(common_ids))
print("New in Advanced:", len(new_in_advanced))
print("Removed from Baseline:", len(removed_from_baseline))
print(f"Top-5 Overlap: {overlap_percentage:.1f}%")

Recommendation Comparison
Baseline recommendations: [947, 824, 737, 749, 888]
Advanced recommendations: [947, 824, 737, 749, 841]

Common recommendations: 4
New in Advanced: 1
Removed from Baseline: 1
Top-5 Overlap: 80.0%


In [39]:
# Advanced Recommendation Validation on 3 User Types

test_users = {
    "Limited History": 20,
    "Medium History": 14,
    "Rich History": 16
}

print("Advanced Recommendation - 3 User Validation")
print("=" * 60)

for user_type, user_id in test_users.items():

    user_interactions = df[df["UserId"] == user_id]
    visited = set(user_interactions["AttractionId"])

    recs = advanced_recommendation(
        user_id=user_id,
        top_n=5
    )

    rec_ids = [rec["AttractionId"] for rec in recs]

    already_visited = set(rec_ids) & visited
    unique_recs = len(set(rec_ids))

    print(f"\n{user_type}")
    print(f"User ID: {user_id}")
    print(f"Interactions: {len(user_interactions)}")
    print("-" * 40)

    for i, rec in enumerate(recs, 1):
        print(
            f"{i}. {rec['Attraction']} "
            f"(ID: {rec['AttractionId']}, Score: {rec['Score']})"
        )

    print(f"Already visited: {len(already_visited)}")
    print(f"Unique recommendations: {unique_recs}")

Advanced Recommendation - 3 User Validation

Limited History
User ID: 20
Interactions: 1
----------------------------------------
1. Sacred Monkey Forest Sanctuary (ID: 640, Score: 0.7)
2. Tegalalang Rice Terrace (ID: 748, Score: 0.3667)
3. Uluwatu Temple (ID: 824, Score: 0.2847)
4. Tanah Lot Temple (ID: 737, Score: 0.2713)
5. Kuta Beach - Bali (ID: 369, Score: 0.2614)
Already visited: 0
Unique recommendations: 5

Medium History
User ID: 14
Interactions: 3
----------------------------------------
1. Tegenungan Waterfall (ID: 749, Score: 0.4339)
2. Water Castle (Tamansari) (ID: 1280, Score: 0.4318)
3. Malang City Square (ID: 937, Score: 0.4119)
4. Waterbom Bali (ID: 841, Score: 0.3928)
5. Tanah Lot Temple (ID: 737, Score: 0.3447)
Already visited: 0
Unique recommendations: 5

Rich History
User ID: 16
Interactions: 10
----------------------------------------
1. Tanah Lot Temple (ID: 737, Score: 0.6738)
2. Sanur Beach (ID: 650, Score: 0.634)
3. Seminyak Beach (ID: 673, Score: 0.6282)
4. Ku

In [40]:
# Baseline vs Advanced Recommendation Coverage

sample_users = df["UserId"].drop_duplicates().sample(
    n=100,
    random_state=42
)

catalog_size = item_full["AttractionId"].nunique()

baseline_recommended = set()
advanced_recommended = set()

for user_id in sample_users:

    baseline_recs = baseline_hybrid_recommendation(
        user_id=user_id,
        top_n=5,
        alpha=0.5
    )

    advanced_recs = advanced_recommendation(
        user_id=user_id,
        top_n=5
    )

    baseline_recommended.update(
        rec["AttractionId"] for rec in baseline_recs
    )

    advanced_recommended.update(
        rec["AttractionId"] for rec in advanced_recs
    )

baseline_unique = len(baseline_recommended)
advanced_unique = len(advanced_recommended)

baseline_coverage = (baseline_unique / catalog_size) * 100
advanced_coverage = (advanced_unique / catalog_size) * 100

coverage_change = advanced_coverage - baseline_coverage
relative_change = (
    (advanced_coverage - baseline_coverage)
    / baseline_coverage * 100
    if baseline_coverage > 0 else 0
)

print("Recommendation Coverage Comparison")
print("=" * 55)

print(f"Users evaluated: {len(sample_users)}")
print(f"Catalog size: {catalog_size} attractions")
print()

print(f"Baseline unique attractions recommended: {baseline_unique}")
print(f"Baseline coverage: {baseline_coverage:.2f}%")
print()

print(f"Advanced unique attractions recommended: {advanced_unique}")
print(f"Advanced coverage: {advanced_coverage:.2f}%")
print()

print(f"Coverage improvement: +{coverage_change:.2f} percentage points")
print(f"Relative coverage increase: +{relative_change:.1f}%")

Recommendation Coverage Comparison
Users evaluated: 100
Catalog size: 1698 attractions

Baseline unique attractions recommended: 23
Baseline coverage: 1.35%

Advanced unique attractions recommended: 21
Advanced coverage: 1.24%

Coverage improvement: +-0.12 percentage points
Relative coverage increase: +-8.7%


In [41]:
# Recommendation Quality Evaluation
# Leave-One-Out Hit Rate @ 5

def evaluate_recommender(recommendation_function, users, top_n=5):

    hits = 0
    evaluated = 0

    for user_id in users:

        user_history = df[df["UserId"] == user_id]

        # Need at least 2 interactions so one can be held out
        if len(user_history) < 2:
            continue

        # Hold out the user's last interaction
        test_attraction = user_history.iloc[-1]["AttractionId"]

        # Get recommendations
        recs = recommendation_function(
            user_id=user_id,
            top_n=top_n
        )

        recommended_ids = {
            rec["AttractionId"]
            for rec in recs
        }

        if test_attraction in recommended_ids:
            hits += 1

        evaluated += 1

    hit_rate = hits / evaluated if evaluated > 0 else 0

    return {
        "Evaluated Users": evaluated,
        "Hits": hits,
        "Hit Rate": hit_rate
    }


# Use the same 100-user sample
baseline_quality = evaluate_recommender(
    baseline_hybrid_recommendation,
    sample_users,
    top_n=5
)

advanced_quality = evaluate_recommender(
    advanced_recommendation,
    sample_users,
    top_n=5
)

print("Recommendation Quality Evaluation")
print("=" * 55)

print("\nBaseline Hybrid")
print(f"Evaluated Users: {baseline_quality['Evaluated Users']}")
print(f"Hits: {baseline_quality['Hits']}")
print(f"Hit Rate@5: {baseline_quality['Hit Rate']:.4f}")

print("\nAdvanced Recommendation")
print(f"Evaluated Users: {advanced_quality['Evaluated Users']}")
print(f"Hits: {advanced_quality['Hits']}")
print(f"Hit Rate@5: {advanced_quality['Hit Rate']:.4f}")

print("\nComparison")
print(
    f"Baseline Hit Rate@5: "
    f"{baseline_quality['Hit Rate']:.4f}"
)

print(
    f"Advanced Hit Rate@5: "
    f"{advanced_quality['Hit Rate']:.4f}"
)

print(
    f"Difference: "
    f"{advanced_quality['Hit Rate'] - baseline_quality['Hit Rate']:+.4f}"
)

Recommendation Quality Evaluation

Baseline Hybrid
Evaluated Users: 23
Hits: 0
Hit Rate@5: 0.0000

Advanced Recommendation
Evaluated Users: 23
Hits: 0
Hit Rate@5: 0.0000

Comparison
Baseline Hit Rate@5: 0.0000
Advanced Hit Rate@5: 0.0000
Difference: +0.0000


In [42]:
# Proper Leave-One-Out Evaluation
# The last interaction is hidden before generating recommendations.

def evaluate_leave_one_out(recommendation_function, users, top_n=5):

    global df

    original_df = df.copy()

    hits = 0
    evaluated = 0

    for user_id in users:

        user_history = original_df[
            original_df["UserId"] == user_id
        ].copy()

        if len(user_history) < 2:
            continue

        # Hold out the last interaction
        test_row = user_history.iloc[-1]
        test_attraction = test_row["AttractionId"]

        # Remove the held-out interaction
        df = original_df.drop(index=test_row.name).copy()

        # Generate recommendations using reduced history
        recs = recommendation_function(
            user_id=user_id,
            top_n=top_n
        )

        recommended_ids = {
            rec["AttractionId"]
            for rec in recs
        }

        if test_attraction in recommended_ids:
            hits += 1

        evaluated += 1

    # Restore original dataframe
    df = original_df

    hit_rate = hits / evaluated if evaluated > 0 else 0

    return {
        "Evaluated Users": evaluated,
        "Hits": hits,
        "Hit Rate@5": hit_rate
    }


baseline_loo = evaluate_leave_one_out(
    baseline_hybrid_recommendation,
    sample_users,
    top_n=5
)

advanced_loo = evaluate_leave_one_out(
    advanced_recommendation,
    sample_users,
    top_n=5
)

print("Proper Leave-One-Out Recommendation Evaluation")
print("=" * 60)

print("\nBaseline Hybrid")
print(f"Evaluated Users: {baseline_loo['Evaluated Users']}")
print(f"Hits: {baseline_loo['Hits']}")
print(f"Hit Rate@5: {baseline_loo['Hit Rate@5']:.4f}")

print("\nAdvanced Recommendation")
print(f"Evaluated Users: {advanced_loo['Evaluated Users']}")
print(f"Hits: {advanced_loo['Hits']}")
print(f"Hit Rate@5: {advanced_loo['Hit Rate@5']:.4f}")

print("\nComparison")
print(
    f"Baseline Hit Rate@5: "
    f"{baseline_loo['Hit Rate@5']:.4f}"
)

print(
    f"Advanced Hit Rate@5: "
    f"{advanced_loo['Hit Rate@5']:.4f}"
)

print(
    f"Difference: "
    f"{advanced_loo['Hit Rate@5'] - baseline_loo['Hit Rate@5']:+.4f}"
)

Proper Leave-One-Out Recommendation Evaluation

Baseline Hybrid
Evaluated Users: 23
Hits: 13
Hit Rate@5: 0.5652

Advanced Recommendation
Evaluated Users: 23
Hits: 12
Hit Rate@5: 0.5217

Comparison
Baseline Hit Rate@5: 0.5652
Advanced Hit Rate@5: 0.5217
Difference: -0.0435


In [43]:
# Final Recommendation Evaluation Summary

recommendation_results = pd.DataFrame([
    {
        "System": "Baseline Hybrid",
        "Top-5 Overlap": "—",
        "Unique Attractions (100 Users)": baseline_unique,
        "Coverage (%)": round(baseline_coverage, 2),
        "Leave-One-Out Hit Rate@5 (%)": round(
            baseline_loo["Hit Rate@5"] * 100, 2
        )
    },
    {
        "System": "Advanced Recommendation",
        "Top-5 Overlap": "80.0%",
        "Unique Attractions (100 Users)": advanced_unique,
        "Coverage (%)": round(advanced_coverage, 2),
        "Leave-One-Out Hit Rate@5 (%)": round(
            advanced_loo["Hit Rate@5"] * 100, 2
        )
    }
])

print("Recommendation System Evaluation Summary")
print("=" * 70)

display(recommendation_results)

Recommendation System Evaluation Summary


,System,Top-5 Overlap,Unique Attractions (100 Users),Coverage (%),Leave-One-Out Hit Rate@5 (%)
0,Baseline Hybrid,—,23,1.35,56.52
1,Advanced Recommendation,80.0%,21,1.24,52.17


In [45]:
# Overall Model Evaluation Comparison

overall_results = pd.DataFrame([
    {
        "Task": "Rating Regression",
        "Model": "XGBoost",
        "Primary Metric": "R²",
        "Primary Score": round(regression_results["R2"], 4),
        "Additional Metrics": (
            f"MAE={regression_results['MAE']:.4f}, "
            f"RMSE={regression_results['RMSE']:.4f}"
        )
    },
    {
        "Task": "Visit Mode Classification",
        "Model": "XGBoost",
        "Primary Metric": "Macro F1",
        "Primary Score": round(classification_results["Macro F1"], 4),
        "Additional Metrics": (
            f"Accuracy={classification_results['Accuracy']:.4f}, "
            f"Macro Precision={classification_results['Precision']:.4f}, "
            f"Macro Recall={classification_results['Recall']:.4f}"
        )
    },
    {
        "Task": "Recommendation",
        "Model": "Baseline Hybrid",
        "Primary Metric": "Leave-One-Out Hit Rate@5",
        "Primary Score": round(
            baseline_loo["Hit Rate@5"], 4
        ),
        "Additional Metrics": (
            f"Coverage={baseline_coverage:.2f}%, "
            f"Unique Attractions={baseline_unique}"
        )
    },
    {
        "Task": "Recommendation",
        "Model": "Advanced Recommendation",
        "Primary Metric": "Leave-One-Out Hit Rate@5",
        "Primary Score": round(
            advanced_loo["Hit Rate@5"], 4
        ),
        "Additional Metrics": (
            f"Coverage={advanced_coverage:.2f}%, "
            f"Unique Attractions={advanced_unique}, "
            f"Top-5 Overlap=80.0%"
        )
    }
])

print("Overall Model Evaluation Comparison")
print("=" * 80)

display(overall_results)

Overall Model Evaluation Comparison


,Task,Model,Primary Metric,Primary Score,Additional Metrics
0,Rating Regression,XGBoost,R²,0.1253,"MAE=0.7105, RMSE=0.9077"
1,Visit Mode Classification,XGBoost,Macro F1,0.3187,"Accuracy=0.4013, Macro Precision=0.3278, Macro..."
2,Recommendation,Baseline Hybrid,Leave-One-Out Hit Rate@5,0.5652,"Coverage=1.35%, Unique Attractions=23"
3,Recommendation,Advanced Recommendation,Leave-One-Out Hit Rate@5,0.5217,"Coverage=1.24%, Unique Attractions=21, Top-5 O..."


# Final Evaluation Conclusions

## Overall Findings

- The XGBoost regression model achieved a held-out R² of 0.1253, with an MAE of 0.7105 and RMSE of 0.9077.
- The XGBoost classification model achieved an accuracy of 0.4013 and a Macro F1-score of 0.3187. The relatively low Macro F1 reflects the difficulty of predicting the imbalanced VisitMode classes.
- The Baseline Hybrid Recommendation System achieved a Leave-One-Out Hit Rate@5 of 56.52% on the evaluated users.
- The Advanced Recommendation System achieved a Leave-One-Out Hit Rate@5 of 52.17% on the same evaluation set.
- The Advanced system retained 4 out of 5 recommendations for the tested user compared with the baseline, resulting in an 80% Top-5 overlap.
- In the 100-user coverage evaluation, the Baseline system recommended 23 unique attractions (1.35% of the 1,698-attraction catalog), while the Advanced system recommended 21 unique attractions (1.24%).
- Therefore, the current offline evaluation does not demonstrate an objective performance improvement of the Advanced Recommendation System over the Baseline Hybrid System.
- The Advanced system nevertheless provides a more flexible recommendation framework by incorporating collaborative filtering, content similarity, popularity, and user-history-based weighting.

## Key Limitations

- The recommendation evaluation was performed on a relatively small set of users with sufficient interaction history.
- The offline leave-one-out evaluation uses historical interaction data and should be interpreted as a diagnostic measure rather than a complete real-world relevance assessment.
- The current interaction data covers only 30 attractions, while the complete catalog contains 1,698 attractions, which limits collaborative-filtering and popularity coverage.
- Further evaluation with larger user samples, stronger temporal hold-out strategies, and real user feedback would be required to establish whether the Advanced system provides better recommendation quality.

## Final Model Selection

For the current dataset and evaluation results, the Baseline Hybrid Recommendation System provides the stronger measured offline recommendation performance. The Advanced Recommendation System is retained as an extended recommendation framework for further experimentation and future improvement.